# Path-AGNN-Cox: reproducible patient-specific pathway rewiringInteractive quickstart for the paper "Path-AGNN-Cox: a reproducible statistical framework for testing patient-specific pathway rewiring in cancer survival analysis".> Note: this notebook runs on synthetic data so it works anywhere with a CPU. Replace with the real processed matrices from the repository to reproduce the manuscript results.

## 1. InstallOnce the package is published, the simplest install is:```pip install path-agnn-cox   # after PyPI release```Until then, install from the repository (edit the URL once the public repo exists):

In [ ]:
# !git clone https://github.com/wangzhipeng-1/Path-AGNN-Cox.git path_agnn_repo# %cd path_agnn_repo# !pip install -r requirements.txtprint("Install step: uncomment after the public repository is released.")

## 2. Synthetic demo (no real data)The same pipeline used for the manuscript: pathway subgraphs -> adaptive attention -> Cox head.

In [ ]:
import sys, osimport numpy as npimport torchsys.path.insert(0, os.path.abspath('.'))from path_agnn_cox.pathway import load_gmt, build_pathway_adjacencyfrom path_agnn_cox.models import PathAGNNCoxfrom path_agnn_cox.train import train_model, predict_riskfrom path_agnn_cox.evaluate import full_report# synthetic pathway prior (3 pathways x 20 genes)rng = np.random.default_rng(0)genes = ['G' + str(i) for i in range(60)]pathway_dict = {'PATH_A': genes[0:20], 'PATH_B': genes[20:40], 'PATH_C': genes[40:60]}with open('_tmp.gmt', 'w') as fh:    for pid, gs in pathway_dict.items():        fh.write(pid + '\tna\t' + '\t'.join(gs) + '\n')pathway_dict = load_gmt('_tmp.gmt')# synthetic cohort: 300 patients, 60 genes, 35% eventsn = 300X = rng.normal(0, 1, (n, len(genes)))z = X[:, :20].mean(1) * 0.8 + X[:, 20:40].mean(1) * 0.5time = np.exp(2.5 - 0.7 * z + rng.normal(0, 0.6, n))time = np.clip(time, 0.1, None)event = (rng.random(n) < 0.35).astype(int)time = np.where(event == 1, time, np.minimum(time, rng.uniform(1, 5, n)))adj, mem, gene_order = build_pathway_adjacency(genes, pathway_dict)Xf = X[:, [genes.index(g) for g in gene_order]]ids = torch.tensor([list(mem.columns).index(mem.loc[g].idxmax()) for g in gene_order])print('pathway adjacency:', adj.shape, 'genes:', len(gene_order))

In [ ]:
idx = rng.permutation(n)tr, va = idx[:240], idx[240:]model = PathAGNNCox(n_genes=len(gene_order), adj=torch.tensor(adj), pathway_ids=ids)train_model(model, Xf[tr], time[tr], event[tr], Xf[va], time[va], event[va],            epochs=60, patience=10, lambda_sparse=0.001, lambda_consist=0.1)risk = predict_risk(model, Xf[va])print('Validation C-index:', round(full_report(risk, time[va], event[va])['c_index'], 3))import os as _osif _os.path.exists('_tmp.gmt'):    _os.remove('_tmp.gmt')

## 3. Rewiring test (the statistical framework)Extract per-patient edge weights, split by risk strata, and run the pathway-level between-stratum test with BH-FDR control -- the core output of the manuscript.

In [ ]:
import numpy as npfrom benchmark.rewiring_analysis import pathway_level_testmodel.eval()with torch.no_grad():    r, alpha, src, dst = model(torch.tensor(Xf[va], dtype=torch.float32), return_alpha=True)risk_v = r.numpy().ravel()med = np.median(risk_v)hi = np.where(risk_v > med)[0]lo = np.where(risk_v <= med)[0]memc = mem.loc[[g for g in gene_order if g in mem.index]]pw = pathway_level_test(alpha.numpy(), hi, lo, src.numpy(), dst.numpy(),                        np.array([g for g in gene_order if g in mem.index]), memc)print(pw[['pathway', 'n_edges', 'z', 'p', 'q']].head(3).to_string(index=False))print('significant (q<0.05):', int((pw['q'] < 0.05).sum()))

## 4. Reproducing the manuscript- Benchmark: `python -m benchmark.run_benchmark --datasets LUAD,BRCA --models path_agnn_cox`- Figures: `python manuscript/make_figures.py`- Manuscript render + audit: `python manuscript/render_manuscript.py` and `python manuscript/check_formatting.py`Repository: [GitHub](https://github.com/YOUR-ORG/Path-AGNN-Cox) · PyPI: `path-agnn-cox` (placeholder)